In [4]:
%matplotlib widget
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from image_utils import load_rdb, save_rgb_png
from android_capture import ScreenCapture
import time
import image_utils
import ocr_cards
import matplotlib.pyplot as plt
from android_capture import ScreenCapture
import time
import image_utils
import ocr_cards
import matplotlib.pyplot as plt
import get_cards_tablet
from image_assets import ImageAssets

In [5]:
scr_taker = None

In [6]:
del scr_taker

scr_taker = ScreenCapture(save_folder="/media/maxim/T7/frames/")
while True:
    scr_taker.get_screen()
    time.sleep(0.5)

[ WARN:0@8.935] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


RuntimeError: Could not open /dev/video2

In [ ]:
# capturer.close()

In [ ]:
with ScreenCapture() as capturer:
    img_np = capturer.get_screen()

        
img_path = "frames_tablet/frame.png"
# img_np = image_utils.load_rdb(img_path)
image_utils.save_rgb_png(img_np, img_path)
img_np = image_utils.load_rdb(img_path)

In [ ]:
img_path = "frames_tablet/pink_ad.png"
img_np = image_utils.load_rdb(img_path)

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(8, 4))
f.tight_layout()
ax.imshow(img_np)

In [ ]:
ad_cross = ImageAssets.ad_cross
matches, n_matches = image_utils.find_subimages(
    img_np, {"cross": ad_cross}, threshold=0.6
)
mid_positions = []

In [ ]:
pink_cross = img_np[1092:1113, 1792:1813]

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(8, 4))
f.tight_layout()
ax.imshow(pink_cross)

In [ ]:
img_np[956:971, 1721:1736].shape

In [ ]:
image_utils.save_rgb_png(pink_cross, "ui_elements/ad_cross_pink.png")

In [ ]:
list(itertools.chain(*(a for a in [[1,3],[2]])))

In [ ]:
from itertools import combinations_with_replacement, product
from collections import Counter, defaultdict
from math import factorial
from collections.abc import Iterable



def multiset_with_perm_counts(n: int, k: int):
    """
    Generate all size-k selections from n elements (with replacement),
    unique up to ordering, and for each yield:
        (values_tuple, number_of_distinct_permutations)

    Elements are:
      - 0..n-1  if one_based=False
      - 1..n    if one_based=True
    """
    if isinstance(n, Iterable):
        elems = n
    else:
        elems = list(range(n))

    for combo in combinations_with_replacement(elems, k):
        counts = Counter(combo).values()
        denom = 1
        for c in counts:
            denom *= factorial(c)
        num_perms = factorial(k) // denom
        yield combo, num_perms

In [ ]:
combinations_with_counts = dict()

for i in range(1, 8):
    combinations_with_counts[i] = [
        (comb, nc) for comb, nc in multiset_with_perm_counts(range(2, 12), i)
    ]

In [ ]:
combinations_with_counts[3]

In [ ]:
import numpy as np


for i, data in combinations_with_counts.items():
    np_data = []
    for comb, count in data:
        np_data.append(list(comb) + [count])
    np.savetxt(f"combinations/combinations_with_counts_{i}.csv", np.array(np_data), fmt="%.0f")

In [ ]:
from blackjack.hand import ValueOnlyHand 


def _dealer_stand(
    dealer_hand: ValueOnlyHand, dealer_hit_soft_17: bool
):
    dealer_stand = False
    best_value = dealer_hand.get_best_value()
    if best_value is None:
        return True # bust
    
    if dealer_hit_soft_17:
        if best_value > 17:
            dealer_stand = True
        elif best_value == 17 and not dealer_hand.is_soft_17():
            dealer_stand = True
    else:
        if best_value >= 17:
            dealer_stand = True
    return dealer_stand



def generate_and_save_combs_with_counts(i: int):
    combinations_with_counts = list(multiset_with_perm_counts(range(2, 12), i))
    np_data = []
    for comb, count in combinations_with_counts:
        np_data.append(list(comb) + [count])
    np.savetxt(f"combinations/combinations_with_counts_{i}.csv", np.array(np_data), fmt="%.0f")
    
    for dealer_hit_soft_17 in [False, True]:
        realistic_combinations = defaultdict(int)
        for comb, count in combinations_with_counts:
            hand = ValueOnlyHand()
            for v in comb:
                if _dealer_stand(hand, dealer_hit_soft_17=dealer_hit_soft_17):
                    break
                hand.add_card(v)
            realistic_combinations[tuple(hand.cards)] += count

        real_data = []
        for comb, count in realistic_combinations.items():
            real_data.append(list(comb) + [count])

        decision_17_str = "h17" if dealer_hit_soft_17 else "s17"
        fname = f"combinations/realistic_comb_with_counts_{decision_17_str}_{i}.csv"
        with open(fname, "w") as f:
            for row in real_data:
                f.write(" ".join([str(r) for r in row]) + "\n")


def load_combinations_with_counts(i: int):
    data = np.loadtxt(f"combinations/combinations_with_counts_{i}.csv", dtype=int)
    combinations = [tuple(row[:-1].tolist()) for row in data]
    counts = data[:, -1].tolist()
    return list(zip(combinations, counts))


In [ ]:
_dealer_stand(ValueOnlyHand([10,10]), dealer_hit_soft_17=False)

In [ ]:
for i in range(1, 10):
    generate_and_save_combs_with_counts(i)

In [ ]:
from blackjack.hand import ValueOnlyHand

In [ ]:
def _dealer_stand(
    dealer_hand: ValueOnlyHand, dealer_hit_soft_17: bool
):
    dealer_stand = False
    best_value = dealer_hand.get_best_value()
    if best_value is None:
        return True # bust
    
    if dealer_hit_soft_17:
        if best_value > 17:
            dealer_stand = True
        elif best_value == 17 and not dealer_hand.is_soft_17():
            dealer_stand = True
    else:
        if best_value >= 17:
            dealer_stand = True
    return dealer_stand

In [ ]:
realistic_combinations = defaultdict(int)
dealer_hit_soft_17 = False

k = 7

for comb, count in combinations_with_counts[k]:
    hand = ValueOnlyHand()
    finished_early = False
    for v in comb:
        if _dealer_stand(hand, dealer_hit_soft_17=dealer_hit_soft_17):
            finished_early = True
            break
        hand.add_card(v)
    realistic_combinations[tuple(hand.cards)] += count

In [ ]:
len(realistic_combinations), len(combinations_with_counts[k])

In [ ]:
realistic_combinations